In [3]:
import os
import pandas as pd
from nilearn.glm.second_level import SecondLevelModel
from nilearn.plotting import plot_stat_map
from nilearn.glm.thresholding import threshold_img
from nilearn.image import load_img

In [4]:
contrast_names = [
    "bra_vs_0",
    "bra_vs_others",
    "so_vs_0",
    "so_vs_others",
    "fi_vs_0",
    "fi_vs_others",
    "ru_vs_0",
    "ru_vs_others",
    'black_vs_white',
    'white_vs_black',
    'others_vs_fi',
    'warm_vs_cold',
    'cold_vs_warm',
]


In [5]:
base_dir = "/scratch/nbe/prevent/Baran/Codes/beta_maps"
output_dir = "/scratch/nbe/prevent/Baran/Codes/group_results"
os.makedirs(output_dir, exist_ok=True)

subjects = [f"sub-{i:02}" for i in range(1, 43)]


In [6]:

for contrast_name in contrast_names: 
    contrast_maps = []
    
    for sub in subjects:
        path = os.path.join(base_dir, sub, f"{contrast_name}.nii.gz")
        if os.path.exists(path):
            contrast_maps.append(path)
        else:
            print(f"missing: {path}")
    
  
    # Simple design matrix
    design_matrix = pd.DataFrame({'intercept': [1] * len(contrast_maps)})

    # Run 2nd-level model
    slm = SecondLevelModel()
    slm = slm.fit(contrast_maps, design_matrix=design_matrix)
    z_map = slm.compute_contrast('intercept', output_type='z_score')

    # z_thr, used_height = threshold_stats_img(
    #     z_map,
    #     alpha=0.05,                
    #     height_control='fdr',      
    #     cluster_threshold=3,       # remove clusters smaller than 3 voxels
    #     two_sided=True           
    # )

    z_thr = threshold_img(z_map, threshold=3.0, cluster_threshold=3, two_sided=True)

    # Save
    z_path = os.path.join(output_dir, f"{contrast_name}_second_level_zmap.nii.gz")
    z_thr.to_filename(z_path)
    

    # Plot
    plot_stat_map(
        z_thr,
        title=f"Second-level: {contrast_name}",
        threshold=3,
        symmetric_cbar=True,
        display_mode='ortho',
        output_file=os.path.join(output_dir, f"{contrast_name}.png")
    )

    print(f"Saved z-map and plot for {contrast_name}")

/tmp/ipykernel_780617/323180957.py:28: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  z_thr = threshold_img(z_map, threshold=3.0, cluster_threshold=3, two_sided=True)


Saved z-map and plot for fi_vs_others


/tmp/ipykernel_780617/323180957.py:28: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  z_thr = threshold_img(z_map, threshold=3.0, cluster_threshold=3, two_sided=True)


Saved z-map and plot for black_vs_white
Saved z-map and plot for white_vs_black
Saved z-map and plot for others_vs_fi
Saved z-map and plot for warm_vs_cold
Saved z-map and plot for cold_vs_warm
Saved z-map and plot for comp_vs_non
Saved z-map and plot for non_vs_comp
